# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

In [3]:
"""
Research question:
Using a page's traffic history, staleness, and position trend, can we produce a ranked
action score that identifies which pages are the best candidates for content refresh --
and does a trained model meaningfully outperform a simple hand-written baseline rule at
flagging the true top opportunities?

Decision this supports: which pages a content editor should prioritize for refresh this
month, and how much confidence to put behind that prioritization.

Unit of analysis: one page (one row of content_refresh_anonymized.csv).
Output: a ranked refresh-opportunity score per page (higher = higher priority).
Lane: Refresh / Content Opportunity Scoring.
"""
print("Research question set -- see the note above.")

Research question set -- see the note above.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [2]:
import pandas as pd

# Data source: data/raw/content_refresh_anonymized.csv -- a 30,000-row anonymized
# teaching slice of the FlyRank warehouse. One row per page, metrics aggregated over a
# trailing 90-day window. 32 pseudonymized clients.
#
# Deliberately excluded as model inputs:
#   - trend_direction / trend_pct  -> these ARE the label (is_declining_label is derived
#     from trend_direction), so using them as features would be leakage.
#   - content_id / client_id       -> identifiers; used only for the client-aware
#     train/test split, never as features.
#   - provider_used / model_used   -> which AI wrote the article isn't a safe causal
#     signal for real-world search performance.

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("Rows, columns:", df.shape)
df.head()

Rows, columns: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [4]:
import json

# Label: is_declining_label = 1 when trend_direction == "down", else 0
#   (trend_direction is a threshold rule on 30-day impression change -- see data dictionary).
#
# Baseline (transparent, hand-written -- scripts/02_baseline_score.py):
#   baseline_refresh_score = 0.40 * visibility_score       (log impressions, percentile-ranked)
#                           + 0.30 * freshness_risk_score   (days since last update, percentile-ranked)
#                           + 0.25 * position_opportunity_score (good position + visible)
#                           + 0.05 * depth_gap_score         (thin content + visible)
#   No fitted weights -- every term is a readable, honest rule.
#
# Model: random_forest trained on numeric + one-hot categorical features
#   (scripts/03_train_model.py), validated with a CLIENT-HOLDOUT split -- 20% of *clients*
#   held out entirely, so the model is never tested on a client it trained on.

feature_meta = json.load(open("../../data/processed/feature_metadata.json"))
baseline_meta = json.load(open("../../data/processed/baseline_metadata.json"))
model_results = json.load(open("../../outputs/model_results.json"))

print("Target definition:", feature_meta["target_definition"])
print("Prepared rows:", feature_meta["prepared_rows"], "| Declining rate:", round(feature_meta["declining_rate"], 3))
print("Baseline formula weights:", baseline_meta["score_formula"])
print("Split strategy:", model_results["split_strategy"],
      "->", model_results["train_rows"], "train /", model_results["test_rows"], "test rows")
print("Numeric features:", model_results["model_numeric_features"])
print("Categorical features:", model_results["model_categorical_features"])
print()
print("Leakage check -- none of these appear in the feature lists above:",
      "trend_direction, trend_pct, content_id, client_id")

Target definition: trend_direction == 'down'
Prepared rows: 30000 | Declining rate: 0.542
Baseline formula weights: {'depth_gap_score': 0.05, 'freshness_risk_score': 0.3, 'position_opportunity_score': 0.25, 'visibility_score': 0.4}
Split strategy: client_holdout -> 27675 train / 2325 test rows
Numeric features: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
Categorical features: ['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']

Leakage check -- none of these appear in the feature lists above: trend_direction, trend_pct, content_id, client_id


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [6]:
# Both the baseline rule and the random_forest model are scored on the SAME held-out
# client-holdout test split (2,325 rows from clients never seen in training).

baseline = model_results["baseline"]
rf = model_results["models"]["random_forest"]
base_rate = model_results["target_positive_rate"]

print(f"Base rate (share of pages actually declining): {base_rate:.3f}")
print()
print(f"{'Metric':<16}{'Baseline rule':>15}{'Random forest':>16}")
for label, key in [
    ("Precision@20", "precision_at_20"),
    ("Precision@50", "precision_at_50"),
    ("Precision@100", "precision_at_100"),
    ("ROC AUC", "roc_auc"),
]:
    b = baseline[f"baseline_{key}"]
    m = rf[key]
    print(f"{label:<16}{b:>15.3f}{m:>16.3f}")

print()
print("The model roughly TRIPLES the baseline's precision@50 (0.24 -> 0.68), and is well")
print("above the 0.542 base rate -- the baseline alone barely beats random guessing at K=50.")

Base rate (share of pages actually declining): 0.542

Metric            Baseline rule   Random forest
Precision@20              0.150           0.700
Precision@50              0.240           0.680
Precision@100             0.360           0.700
ROC AUC                   0.627           0.747

The model roughly TRIPLES the baseline's precision@50 (0.24 -> 0.68), and is well
above the 0.542 base rate -- the baseline alone barely beats random guessing at K=50.


## 5. Limitations

*What this work cannot claim.*

In [7]:
# Limitations, stated plainly:
#
# - Observational, not causal. The model finds pages that LOOK like they're declining;
#   it does not explain WHY, and it cannot promise that refreshing a page will fix anything.
# - The label is a proxy. "Declining" means "impressions dropped >20% vs the prior 30 days"
#   (trend_direction's own rule) -- a measurable proxy, not an editor's ground-truth judgment.
# - Small, single-slice evaluation. Precision@50 of 0.68 is measured on a 30k-row, 32-client
#   teaching slice with a client-holdout split -- directional and decision-support, not a
#   guarantee that holds identically on the full warehouse or on clients never seen at all.
# - Zero/blank edge cases. avg_position, ctr, engagement_rate, and scroll_rate all have
#   documented zero/blank cases (see data dictionary) that were zero-filled upstream -- this
#   can under-score pages with genuinely missing measurement vs. pages with genuinely poor
#   performance. We did not separately verify this distinction in the current run.
# - No causal language used below: results are described as "observed", "measured", or
#   "directional", never as proof that an action will produce an outcome.

print("Limitations documented above.")

Limitations documented above.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [8]:
# Action playbook -- how an editor would use this tomorrow:
#   refresh_and_review_ctr         -> update content AND check title/meta for CTR issues
#   refresh_and_review_engagement  -> update content AND check on-page engagement issues
#   refresh                        -> straightforward content refresh
#   expand_and_refresh             -> page is thin (short) as well as stale -- expand it
#   monitor                        -> no action now, keep watching

queue = pd.read_csv("../../outputs/refresh_queue.csv")

action_counts = queue["suggested_action"].value_counts()
print("Action mix across all 30,000 scored pages:")
print(action_counts.to_string())
print()

top10 = queue.head(10)[[
    "final_rank", "final_refresh_score", "suggested_action",
    "final_reason_codes", "confidence", "impressions_90d", "avg_position", "trend_direction",
]]
top10

Action mix across all 30,000 scored pages:
suggested_action
monitor                          13063
refresh                           8211
refresh_and_review_ctr            6657
refresh_and_review_engagement     1987
expand_and_refresh                  82



,final_rank,final_refresh_score,suggested_action,final_reason_codes,confidence,impressions_90d,avg_position,trend_direction
0,1,81.887229,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...,high,12834,6.8,down
1,2,81.645447,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,high,8064,3.8,down
2,3,81.263671,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,medium,2498,10.1,down
3,4,80.948664,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,high,13790,8.2,down
4,5,80.731581,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,high,1622,3.1,down
5,6,80.685685,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,high,5811,6.4,down
6,7,80.681466,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,medium,3393,3.6,down
7,8,80.244456,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...,high,2655,8.3,down
8,9,80.181563,refresh,declining_with_demand|model_decline_risk|visib...,medium,4366,20.1,down
9,10,80.084814,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,high,2621,12.8,down


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [9]:
from pathlib import Path

# These SVG charts were generated by scripts/04_evaluate_and_export.py during the same
# pipeline run that produced the numbers above -- reused here rather than regenerated,
# so the report and the notebook stay consistent with each other.

chart_dir = Path("../../outputs/charts")
charts = sorted(chart_dir.glob("*.svg"))
print(f"{len(charts)} chart artifacts ready for the deployed paper:")
for chart in charts:
    print(" -", chart.name)

5 chart artifacts ready for the deployed paper:
 - action_mix.svg
 - confidence_mix.svg
 - top_feature_importance.svg
 - top_reason_codes.svg
 - trend_distribution.svg


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.